# 06 — Clock and Time-to-Expiry Effects

**Purpose:** measure the deterministic clock before any regime model is fitted (framework v2.1, Step 2). Every session runs 08:00 → 08:00 UTC, so time of day and time to expiry are **the same variable**:

$$
\tau_t = 1440 - \big((\text{minute of day}_t - 480) \bmod 1440\big) \quad \text{(minutes)}
$$

The only clock variation independent of $\tau_t$ is the calendar $c_t$: weekday/weekend, expiry type (daily, weekly, monthly, quarterly) and daylight saving. With daily expiries alone, an intraday pattern can never be attributed to "time of day" rather than "time to expiry"; the notebook reports it as one clock effect.

| Part | Question | Why it matters |
|---|---|---|
| A | Are the clock variables correct? | They become conditioning inputs of the baseline |
| B | How large and how stable is the intraday volatility and liquidity pattern? | A latent model fitted on raw features would learn the clock as regimes |
| C | How much of volatility variation is clock, day level, or neither; which smooth baseline is enough; does removing it help forecasting? | Specification of $m_x(\tau,c)$, $s_x(\tau,c)$ and of the structural scale $\sigma_{k,t}$ |
| D | Do 0DTE options price the intraday pattern; how do premium, quote support and smile shape change toward expiry? | Option-side baseline; volatility-book timing |
| E | Do weekends and expiry types differ at session level? | Calendar conditioning $c_t$ |

Profiles are estimated **in sample** unless stated otherwise (descriptive). Out-of-sample tests train on the first half of sessions and test on the second. All timestamps are UTC.

## 1. Environment and read-only connection

In [ ]:
from pathlib import Path
import os
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 200)

load_dotenv()
data_root_env = os.getenv("DERIBIT_DATA_ROOT")
if not data_root_env:
    raise RuntimeError("DERIBIT_DATA_ROOT is not defined in the environment/.env file.")
DB_PATH = Path(data_root_env).expanduser().resolve() / "database" / "research.duckdb"
if not DB_PATH.exists():
    raise FileNotFoundError(f"Research database not found: {DB_PATH}")

con = duckdb.connect(str(DB_PATH), read_only=True)
con.execute("SET TimeZone = 'UTC'")
print(f"Research DB: {DB_PATH} | time zone:", con.execute("SELECT current_setting('TimeZone')").fetchone()[0])

PERIODS_PER_YEAR = 365.0 * 24.0 * 12.0
SETTLE_WEIGHT = 1.0 / 3.0

## 2. Load the state table

Requires `time_series_feature_state` schema 0.5.0 or later (clock variables and perpetual integrity gate).

In [ ]:
cols = set(con.execute("DESCRIBE time_series_feature_state").df()["column_name"])
REQUIRED = ["utc_hour", "utc_minute_of_day", "utc_day_of_week", "is_weekend", "expiry_type",
            "us_dst", "eu_dst", "perp_integrity_ok"]
absent = [c for c in REQUIRED if c not in cols]
if absent:
    raise RuntimeError(f"Clock/integrity columns missing {absent}: rebuild with build_features.py (schema >= 0.5.0).")

WANTED = [
    "state_id", "state_time", "underlying", "session_id", "expiry_date", "minutes_to_expiry",
    "return_5m", "realized_vol_1h_annualized", "realized_vol_30m_annualized",
    "log1p_volume_usd", "book_update_count", "futures_spread_bps", "book_obi_20_coherence_noise",
    "option_iv_atm_forward", "fit_atm_iv", "fit_ok", "fit_skew_rel", "fit_curvature_rel", "fit_wrmse",
    "otm_bid_share_z2", "option_atm_quote_ok", "option_shape_quote_ok",
    "perp_integrity_ok", "in_settlement_window",
] + REQUIRED[:-1]
missing = [c for c in WANTED if c not in cols]
df = con.execute(
    f"SELECT {', '.join(dict.fromkeys(c for c in WANTED if c in cols))} FROM time_series_feature_state ORDER BY underlying, state_time"
).df()
df["state_time"] = pd.to_datetime(df["state_time"], utc=True)
if missing:
    print("Missing optional columns:", missing)


def as_bool(s):
    if s.dtype == object:
        s = s.map({"True": True, "False": False, "true": True, "false": False, True: True, False: False})
    return s.fillna(False).astype(bool)


for c in ["perp_integrity_ok", "fit_ok", "option_atm_quote_ok", "option_shape_quote_ok", "in_settlement_window",
          "is_weekend", "us_dst", "eu_dst"]:
    if c in df:
        df[c] = as_bool(df[c])

if "expiry_date" in df:
    df["session_key"] = pd.to_datetime(df["expiry_date"]).dt.strftime("%Y-%m-%d")
else:
    df["session_key"] = ((df["state_time"] - pd.Timedelta(hours=8)).dt.floor("D") + pd.Timedelta(days=1)).dt.strftime("%Y-%m-%d")

df["r"] = pd.to_numeric(df["return_5m"], errors="coerce")
df["r2"] = df["r"] ** 2
df["log_abs_r"] = np.log(df["r"].abs().where(df["r"].abs() > 0))
df["log_updates"] = np.log1p(pd.to_numeric(df.get("book_update_count"), errors="coerce"))
df["log_spread"] = np.log(pd.to_numeric(df.get("futures_spread_bps"), errors="coerce").where(lambda s: s > 0))
df["log_rv1h"] = np.log(pd.to_numeric(df["realized_vol_1h_annualized"], errors="coerce").where(lambda s: s > 0))
df["minute_of_day"] = pd.to_numeric(df["utc_minute_of_day"], errors="coerce")
df["hour"] = pd.to_numeric(df["utc_hour"], errors="coerce").astype("Int64")

ASSETS = sorted(df["underlying"].unique())
CORE = [a for a in ["BTC", "ETH"] if a in ASSETS]
TTE_EDGES = [0, 30, 60, 180, 360, 720, 1080, 1440]
TTE_LABELS = ["≤30m (settle)", "30m–1h", "1–3h", "3–6h", "6–12h", "12–18h", "18–24h"]
df["tte_bucket"] = pd.cut(df["minutes_to_expiry"], bins=TTE_EDGES, labels=TTE_LABELS, right=True)

ret_sessions = sorted(df.loc[df["r"].notna(), "session_key"].unique())
TRAIN_SESSIONS = set(ret_sessions[: len(ret_sessions) // 2])
df["is_train"] = df["session_key"].isin(TRAIN_SESSIONS)
print(f"Rows: {len(df):,} | assets: {ASSETS} | {df['state_time'].min()} → {df['state_time'].max()}")
print(f"Sessions with returns: {len(ret_sessions)} | train: {min(TRAIN_SESSIONS)} → {max(TRAIN_SESSIONS)} "
      f"| test: {min(set(ret_sessions) - TRAIN_SESSIONS)} → {max(ret_sessions)}")

In [ ]:
def eta2(y, groups, min_n=50):
    """Share of variance explained by group means (correlation ratio). Upward bias about (groups-1)/n."""
    d = pd.DataFrame({"y": pd.to_numeric(y, errors="coerce"), "g": groups}).dropna()
    if len(d) < min_n or d["y"].nunique() <= 10:
        return np.nan
    gm = d.groupby("g", observed=True)["y"].transform("mean")
    sst = ((d["y"] - d["y"].mean()) ** 2).sum()
    return float(((gm - d["y"].mean()) ** 2).sum() / sst) if sst > 0 else np.nan


def ols_fit_predict(y_tr, X_tr, X_te):
    A = np.column_stack([np.ones(len(X_tr)), X_tr])
    b, *_ = np.linalg.lstsq(A, y_tr, rcond=None)
    return np.column_stack([np.ones(len(X_te)), X_te]) @ b, b


def oos_r2(y_te, pred, y_tr_mean):
    sst = ((y_te - y_tr_mean) ** 2).sum()
    return float(1 - ((y_te - pred) ** 2).sum() / sst) if sst > 0 else np.nan


def clock_design(d, K=0, hour_dummies=False, weekend=False, weekend_harmonics=0, expiry_types=False):
    """Deterministic clock regressors: Fourier terms of minute of day, hour dummies, calendar terms."""
    m = d["minute_of_day"].to_numpy(float)
    X = {}
    for k in range(1, K + 1):
        X[f"cos{k}"] = np.cos(2 * np.pi * k * m / 1440.0)
        X[f"sin{k}"] = np.sin(2 * np.pi * k * m / 1440.0)
    if hour_dummies:
        h = d["hour"].to_numpy(float)
        for j in range(1, 24):
            X[f"h{j}"] = (h == j).astype(float)
    if weekend:
        w = d["is_weekend"].to_numpy(float)
        X["weekend"] = w
        for k in range(1, weekend_harmonics + 1):
            X[f"wcos{k}"] = w * np.cos(2 * np.pi * k * m / 1440.0)
            X[f"wsin{k}"] = w * np.sin(2 * np.pi * k * m / 1440.0)
    if expiry_types:
        for t in ["weekly", "monthly", "quarterly"]:
            X[f"exp_{t}"] = (d["expiry_type"] == t).to_numpy(float)
    return pd.DataFrame(X, index=d.index)


def session_bootstrap_profile(g, key, B=1000, seed=0):
    """Bootstrap over sessions of E[r^2 | key] / E[r^2]. Rows: replicates; columns: key values."""
    t = g.dropna(subset=["r2"]).pivot_table(index="session_key", columns=key, values="r2",
                                             aggfunc=["sum", "count"], fill_value=0)
    S, C = t["sum"].to_numpy(float), t["count"].to_numpy(float)
    rng = np.random.default_rng(seed)
    W = rng.multinomial(len(S), np.ones(len(S)) / len(S), size=B).astype(float)
    ss, cc = W @ S, W @ C
    prof = (ss / np.where(cc > 0, cc, np.nan)) / (ss.sum(axis=1) / cc.sum(axis=1))[:, None]
    return pd.DataFrame(prof, columns=t["sum"].columns)

# Part A — Clock Variable Audit

## 3. Consistency of the clock variables

Checks: one expiry type per session; $\tau_t$ equal to the formula above; weekend flag equal to ISO day 6–7; daylight-saving flags. The sample (20 Sep → 6 Oct 2026) contains **no daylight-saving transition**: Europe moves on 25 Oct 01:00 UTC, the US on 1 Nov 06:00 UTC (the US equity open moves from 13:30 to 14:30 UTC). Daylight-saving effects therefore cannot be estimated yet; the flags are only verified.

In [ ]:
mod = df["minute_of_day"]
expected_tte = 1440 - ((mod - 480) % 1440)
audit = pd.Series({
    "states": len(df),
    "tte formula mismatches": int((df["minutes_to_expiry"] != expected_tte).sum()),
    "weekend flag mismatches": int((df["is_weekend"] != df["utc_day_of_week"].isin([6, 7])).sum()),
    "sessions with >1 expiry type": int((df.groupby("session_key")["expiry_type"].nunique() > 1).sum()),
    "us_dst values": str(sorted(df["us_dst"].unique().tolist())),
    "eu_dst values": str(sorted(df["eu_dst"].unique().tolist())),
})
display(audit.to_frame("value"))

sessions = df.groupby("session_key").agg(
    expiry_weekday=("session_key", lambda s: pd.Timestamp(s.iloc[0]).day_name()),
    expiry_type=("expiry_type", "first"),
    weekend_share=("is_weekend", "mean"),
    states_with_return=("r", lambda s: int(s.notna().sum())),
)
display(sessions.round(2))
print("Sessions by expiry type (with returns):")
display(sessions[sessions["states_with_return"] > 0]["expiry_type"].value_counts().to_frame("sessions"))

# Part B — Intraday Pattern

## 4. Intraday variance profile

For each asset and UTC hour $h$:

$$
s_h = \frac{\mathbb{E}[r_t^2 \mid \text{hour}_t = h]}{\mathbb{E}[r_t^2]}
$$

$s_h = 2$ means bars in that hour carry twice the average variance. Squared returns are heavy-tailed, so a robust version $\big(\operatorname{med}|r_t| \text{ in } h \,/\, \operatorname{med}|r_t|\big)^2$ is shown next to it, together with $\eta^2$, the share of the variance of $\log|r_t|$ explained by the hour (bias about $23/n$, i.e. under 0.01 here). Returns are index returns, so no perpetual gate is needed.

In [ ]:
prof_rows, robust_rows, eta_rows = [], [], []
for asset, g in df[df["r"].notna()].groupby("underlying"):
    prof_rows.append((g.groupby("hour")["r2"].mean() / g["r2"].mean()).rename(asset))
    med = g["r"].abs().median()
    robust_rows.append(((g.groupby("hour")["r"].apply(lambda s: s.abs().median()) / med) ** 2).rename(asset))
    eta_rows.append({"underlying": asset, "states": len(g), "zero_return_share": float((g["r"] == 0).mean()),
                     "eta2_hour_log_abs_r": eta2(g["log_abs_r"], g["hour"]),
                     "max_over_min_s_h": float(prof_rows[-1].max() / prof_rows[-1].min())})
variance_profile = pd.concat(prof_rows, axis=1)
robust_profile = pd.concat(robust_rows, axis=1)
clock_strength = pd.DataFrame(eta_rows).set_index("underlying")

print("Variance profile s_h (mean r^2 in hour / overall mean r^2):")
display(variance_profile.round(2))
print("Robust profile (median |r| ratio squared):")
display(robust_profile.round(2))
display(clock_strength.round(3))

fig, ax = plt.subplots(figsize=(11, 4))
for a in variance_profile.columns:
    ax.plot(variance_profile.index, variance_profile[a], marker="o", ms=3, lw=1.5 if a in CORE else 0.8, label=a)
ax.axhline(1, color="grey", lw=0.8)
for x, lab in [(8, "expiry 08:00"), (13.5, "US open (DST)"), (20, "US close")]:
    ax.axvline(x, color="grey", ls=":", lw=0.8)
    ax.text(x + 0.1, ax.get_ylim()[1] * 0.95, lab, fontsize=8)
ax.set_xlabel("UTC hour"); ax.set_ylabel("$s_h$"); ax.set_title("Intraday variance profile (index returns)")
ax.legend(ncol=4, fontsize=8)
plt.tight_layout(); plt.show()

## 5. Is the profile stable?

Three checks: (i) bootstrap over **sessions** (not bars, which are dependent) gives a 95% interval for every $s_h$; (ii) profiles estimated on the first and second half of sessions are compared by Spearman correlation across hours; (iii) the profiles of different assets are compared with each other. A clock pattern that is real should be stable across halves and shared by assets.

In [ ]:
boot_rows, split_rows = [], []
for asset, g in df[df["r"].notna()].groupby("underlying"):
    bp = session_bootstrap_profile(g, "hour", B=1000, seed=6)
    lo, hi = bp.quantile(0.025), bp.quantile(0.975)
    boot_rows.append(pd.DataFrame({"underlying": asset, "hour": bp.columns, "s_h": variance_profile[asset].reindex(bp.columns).to_numpy(),
                                   "ci_low": lo.to_numpy(), "ci_high": hi.to_numpy()}))
    a = g[g["is_train"]].groupby("hour")["r2"].mean()
    b = g[~g["is_train"]].groupby("hour")["r2"].mean()
    split_rows.append({"underlying": asset, "spearman_first_vs_second_half": float(a.corr(b, method="spearman")),
                       "hours_ci_above_1": int((lo > 1).sum()), "hours_ci_below_1": int((hi < 1).sum())})
profile_ci = pd.concat(boot_rows, ignore_index=True)
profile_stability = pd.DataFrame(split_rows).set_index("underlying")
display(profile_stability.round(3))
for a in CORE:
    print(f"{a}: s_h with session-bootstrap 95% interval")
    display(profile_ci[profile_ci["underlying"] == a].set_index("hour")[["s_h", "ci_low", "ci_high"]].round(2).T)
print("Spearman correlation of hourly profiles across assets:")
display(variance_profile.corr(method="spearman").round(2))

## 6. Weekday versus weekend

Weekend volatility is compared with weekday volatility at the same hours. The ratio $\mathbb{E}[r^2 \mid \text{weekend}] / \mathbb{E}[r^2 \mid \text{weekday}]$ has a session-bootstrap interval. The 30-minute weekday profile then shows which scheduled times stand out (US equity open 13:30 UTC under US daylight saving, US close 20:00, the 08:00 expiry).

In [ ]:
wk_rows, wk_profiles = [], []
for asset, g in df[df["r"].notna()].groupby("underlying"):
    bp = session_bootstrap_profile(g.assign(wk=g["is_weekend"].map({True: "weekend", False: "weekday"})), "wk", B=1000, seed=7)
    if {"weekend", "weekday"} <= set(bp.columns):
        ratio = bp["weekend"] / bp["weekday"]
        point = g.loc[g["is_weekend"], "r2"].mean() / g.loc[~g["is_weekend"], "r2"].mean()
        wk_rows.append({"underlying": asset, "weekend_over_weekday_var": point,
                        "ci_low": ratio.quantile(0.025), "ci_high": ratio.quantile(0.975),
                        "weekend_states": int(g["is_weekend"].sum())})
    base = g.loc[~g["is_weekend"], "r2"].mean()
    wk_profiles.append(g.groupby(["is_weekend", "hour"])["r2"].mean().div(base).rename(asset))
weekend_ratio = pd.DataFrame(wk_rows).set_index("underlying")
display(weekend_ratio.round(3))
wk_profile = pd.concat(wk_profiles, axis=1).unstack(level=0)
print("Hourly variance relative to the weekday mean (columns: asset, is_weekend):")
display(wk_profile.round(2))

wd = df[df["r"].notna() & ~df["is_weekend"]].copy()
wd["half_hour"] = (wd["minute_of_day"] // 30) * 30
hh = wd.groupby(["underlying", "half_hour"])["r2"].mean().div(wd.groupby("underlying")["r2"].mean(), level="underlying")
hh = hh.unstack(0)
hh.index = [f"{m // 60:02d}:{m % 60:02d}" for m in hh.index]
print("Weekday 30-minute variance profile; the six largest buckets per asset:")
display(pd.DataFrame({a: hh[a].nlargest(6).round(2).reset_index().apply(lambda r: f"{r.iloc[0]} ({r.iloc[1]})", axis=1).to_list()
                      for a in hh.columns}))

## 7. Liquidity and noise clock

Hourly medians of activity, spread and book coherence noise, and $\eta^2$ of hour (24 groups) and of hour × weekend (48 groups). Perpetual features use only states that pass the integrity gate. A feature with large $\eta^2$ needs a clock baseline before it enters any emission.

In [ ]:
LIQ = {"log1p_volume_usd": "log volume", "log_updates": "log book updates", "log_spread": "log spread",
       "book_obi_20_coherence_noise": "coherence noise"}
LIQ = {k: v for k, v in LIQ.items() if k in df}
gated = df[df["perp_integrity_ok"]].copy()
gated["hour_wk"] = gated["hour"].astype(str) + "_" + gated["is_weekend"].astype(str)
rows = []
for asset, g in gated.groupby("underlying"):
    for c, lab in LIQ.items():
        rows.append({"underlying": asset, "feature": lab, "states": int(g[c].notna().sum()),
                     "eta2_hour": eta2(g[c], g["hour"]), "eta2_hour_x_weekend": eta2(g[c], g["hour_wk"]),
                     "eta2_weekend": eta2(g[c], g["is_weekend"])})
liquidity_clock = pd.DataFrame(rows)
display(liquidity_clock.pivot(index="feature", columns="underlying", values="eta2_hour").round(2))
print("hour × weekend:")
display(liquidity_clock.pivot(index="feature", columns="underlying", values="eta2_hour_x_weekend").round(2))

if CORE:
    fig, axes = plt.subplots(1, len(LIQ), figsize=(4 * len(LIQ), 3.2))
    axes = np.atleast_1d(axes)
    for ax, (c, lab) in zip(axes, LIQ.items()):
        for a in ASSETS:
            m = gated[gated["underlying"] == a].groupby("hour")[c].median()
            ax.plot(m.index, m - m.mean(), lw=1.5 if a in CORE else 0.7, label=a)
        ax.set_title(f"{lab} (median by hour, demeaned)", fontsize=9); ax.set_xlabel("UTC hour")
    axes[0].legend(fontsize=7)
    plt.tight_layout(); plt.show()

# Part C — What the Clock Explains

## 8. Clock, day level and the rest

Non-overlapping hourly observations of $\log \mathrm{RV}_{1h}$ (one per hour, at minute 55) are regressed on hour dummies, on session dummies, and on both:

$$
\log \mathrm{RV}_{1h,t} = \alpha_{\text{hour}(t)} + \delta_{\text{session}(t)} + e_t
$$

$R^2_{\text{clock}}$ is variation a deterministic baseline removes; $R^2_{\text{day}}$ is slow, day-level variation (a candidate for slow regimes or volatility clustering); what remains of $1 - R^2_{\text{both}}$ is intraday variation not explained by either, where intraday regimes must live. With 24 hour and about 17 session dummies on about 400 observations per asset, the in-sample $R^2$ is inflated by roughly $40/400 \approx 0.10$; the adjusted $R^2$ is also shown.

In [ ]:
def r2_dummies(y, *factors):
    X = pd.get_dummies(pd.concat(factors, axis=1).astype(str), drop_first=True).astype(float)
    A = np.column_stack([np.ones(len(y)), X.to_numpy()])
    b, *_ = np.linalg.lstsq(A, y.to_numpy(), rcond=None)
    u = y.to_numpy() - A @ b
    r2 = 1 - (u ** 2).sum() / ((y - y.mean()) ** 2).sum()
    k = A.shape[1] - 1
    return r2, 1 - (1 - r2) * (len(y) - 1) / max(len(y) - k - 1, 1)


rows = []
hourly = df[(df["minute_of_day"] % 60 == 55) & df["log_rv1h"].notna()]
for asset, g in hourly.groupby("underlying"):
    if len(g) < 100:
        continue
    y = g["log_rv1h"]
    rc, rca = r2_dummies(y, g["hour"])
    rd, rda = r2_dummies(y, g["session_key"])
    rb, rba = r2_dummies(y, g["hour"], g["session_key"])
    rows.append({"underlying": asset, "hours": len(g), "r2_clock": rc, "adj_r2_clock": rca, "r2_day": rd,
                 "adj_r2_day": rda, "r2_both": rb, "adj_r2_both": rba, "unexplained_adj": 1 - rba})
variance_decomposition = pd.DataFrame(rows).set_index("underlying")
display(variance_decomposition.round(3))

## 9. Does the clock create apparent persistence?

If volatility follows a daily cycle, $|r_t|$ is autocorrelated at lags of hours and peaks again at one day (288 bars), even with no regime at all. Returns are deseasonalized with the in-sample hour × weekend profile:

$$
\tilde r_t = r_t \big/ \sqrt{s_{h(t),\,w(t)}}
$$

and the autocorrelation of $|r_t|$ and $|\tilde r_t|$ is compared on the exact five-minute grid. A drop at 3–12 h lags and at 288 after deseasonalization is persistence that a regime model would otherwise attribute to regimes; what survives is volatility clustering.

In [ ]:
LAGS = [1, 3, 6, 12, 36, 72, 144, 216, 287, 288, 289]
acf_rows, acf_curves = [], {}
for asset, g in df[df["r"].notna()].groupby("underlying"):
    s = g.groupby(["hour", "is_weekend"])["r2"].mean() / g["r2"].mean()
    key = pd.MultiIndex.from_arrays([g["hour"], g["is_weekend"]])
    g = g.assign(r_tilde=g["r"].to_numpy() / np.sqrt(s.reindex(key).to_numpy()))
    grid = pd.date_range(g["state_time"].min(), g["state_time"].max(), freq="5min")
    raw = g.set_index("state_time")["r"].abs().reindex(grid)
    des = g.set_index("state_time")["r_tilde"].abs().reindex(grid)
    row = {"underlying": asset}
    for L in LAGS:
        row[f"raw_{L}"] = raw.autocorr(L)
        row[f"des_{L}"] = des.autocorr(L)
    acf_rows.append(row)
    if asset in CORE:
        acf_curves[asset] = (pd.Series([raw.autocorr(L) for L in range(1, 301)], index=range(1, 301)),
                             pd.Series([des.autocorr(L) for L in range(1, 301)], index=range(1, 301)))
acf_table = pd.DataFrame(acf_rows).set_index("underlying")
for kind in ["raw", "des"]:
    print("|r| autocorrelation" if kind == "raw" else "|r~| autocorrelation (deseasonalized)")
    t = acf_table[[f"{kind}_{L}" for L in LAGS]]
    t.columns = LAGS
    display(t.round(3))
print("95% band for zero autocorrelation ≈ ±", round(1.96 / np.sqrt(df.groupby("underlying")["r"].count().min()), 3))

if acf_curves:
    fig, axes = plt.subplots(1, len(acf_curves), figsize=(6 * len(acf_curves), 3.2), squeeze=False)
    for ax, (a, (raw_c, des_c)) in zip(axes[0], acf_curves.items()):
        ax.plot(raw_c.index, raw_c, lw=1, label="|r|")
        ax.plot(des_c.index, des_c, lw=1, label="|r| deseasonalized")
        ax.axhline(0, color="grey", lw=0.6); ax.axvline(288, color="grey", ls=":", lw=0.8)
        ax.set_title(f"{a}: ACF of absolute returns"); ax.set_xlabel("lag (5-minute bars)"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 10. Which clock baseline is enough? (Step 2 specification)

For each feature $x$, clock baselines of increasing flexibility are fitted on the first half of sessions and evaluated on the second half (out-of-sample $R^2$ against the training mean):

| Model | Regressors |
|---|---|
| F1, F2, F4, F6 | Fourier terms $\cos(2\pi k m/1440), \sin(2\pi k m/1440)$, $k \le K$, $m$ = minute of day |
| H24 | hour dummies (step function) |
| F4+W | F4 + weekend shift + weekend × first two harmonics |
| F4+W+E | F4+W + weekly/monthly/quarterly expiry dummies |

The framework asks for a **smooth and simple** baseline. The preferred specification is the smallest one whose out-of-sample $R^2$ is close to the best. Negative values mean the clock baseline does worse than a constant out of sample. Option features exclude the settlement window and use only states with an accepted quote fit; perpetual features use only integrity-gated states. With few test weekends and only one or two non-daily expiries in each half, the calendar terms are weakly identified.

In [ ]:
FEATURES = {
    "log|r|": ("log_abs_r", None),
    "log RV1h": ("log_rv1h", None),
    "log volume": ("log1p_volume_usd", "perp"),
    "log book updates": ("log_updates", "perp"),
    "log spread": ("log_spread", "perp"),
    "coherence noise": ("book_obi_20_coherence_noise", "perp"),
    "log fit ATM IV": ("log_fit_atm_iv", "fit"),
    "fit skew rel": ("fit_skew_rel", "fit"),
    "fit curvature rel": ("fit_curvature_rel", "fit"),
    "OTM bid share z2": ("otm_bid_share_z2", "opt"),
}
df["log_fit_atm_iv"] = np.log(pd.to_numeric(df.get("fit_atm_iv"), errors="coerce").where(lambda s: s > 0))
BASELINES = {
    "F1": dict(K=1), "F2": dict(K=2), "F4": dict(K=4), "F6": dict(K=6), "H24": dict(hour_dummies=True),
    "F4+W": dict(K=4, weekend=True, weekend_harmonics=2),
    "F4+W+E": dict(K=4, weekend=True, weekend_harmonics=2, expiry_types=True),
}
rows = []
for asset, g0 in df.groupby("underlying"):
    for lab, (col, scope) in FEATURES.items():
        if col not in g0:
            continue
        g = g0
        if scope == "perp":
            g = g[g["perp_integrity_ok"]]
        elif scope == "fit":
            g = g[g["fit_ok"] & ~g["in_settlement_window"]] if "fit_ok" in g else g.iloc[0:0]
        elif scope == "opt":
            g = g[~g["in_settlement_window"]]
        g = g[pd.to_numeric(g[col], errors="coerce").notna()]
        tr, te = g[g["is_train"]], g[~g["is_train"]]
        if len(tr) < 300 or len(te) < 300 or g[col].nunique() <= 10:
            continue
        row = {"underlying": asset, "feature": lab, "n_train": len(tr), "n_test": len(te)}
        for name, spec in BASELINES.items():
            pred, _ = ols_fit_predict(tr[col].to_numpy(float), clock_design(tr, **spec).to_numpy(),
                                      clock_design(te, **spec).to_numpy())
            row[name] = oos_r2(te[col].to_numpy(float), pred, tr[col].mean())
        rows.append(row)
baseline_oos = pd.DataFrame(rows)
if len(baseline_oos):
    print("Out-of-sample R^2 of clock baselines, median across assets:")
    display(baseline_oos.groupby("feature")[list(BASELINES)].median().round(3))
    print("By asset:")
    display(baseline_oos.set_index(["feature", "underlying"])[list(BASELINES)].round(3))

## 11. Does deseasonalizing improve the volatility forecast?

Same target and baseline as Notebook 05 §6: next-30-minute realized volatility,

$$
\log \mathrm{RV}^{fwd}_t = \tfrac12 \log\!\Big(\tfrac{\mathrm{PERIODS\_PER\_YEAR}}{6}\textstyle\sum_{i=1}^{6} r_{t+i}^2\Big)
$$

- **M0 (HAR):** $\log \mathrm{RV}_{1h,t}$, $\log \mathrm{RV}_{30m,t}$, $\log|r_t|$.
- **M0+clock:** M0 + $\log \bar s^{\,fwd}_t$ + $\log \bar s^{\,1h}_t$, where $\bar s^{\,fwd}_t = \frac16\sum_{i=1}^{6} s_{h(t+i),w(t+i)}$ is the expected seasonal variance of the forecast window and $\bar s^{\,1h}_t$ that of the trailing hour, both from the **training** profile.
- **M0+weekend:** M0 + weekend dummy.

If the clock matters, the coefficient on $\log \bar s^{\,fwd}$ is near $\tfrac12$ (the target is a log volatility) and the out-of-sample $R^2$ rises. All states, no perpetual gate (index-only inputs).

In [ ]:
def grid_shift(d, column, k):
    out = pd.Series(np.nan, index=d.index)
    for _, g in d.groupby("underlying"):
        g = g.sort_values("state_time")
        v = pd.to_numeric(g[column], errors="coerce").shift(k)
        ok = (g["state_time"] - g["state_time"].shift(k)).abs() == pd.Timedelta(minutes=5 * abs(k))
        out.loc[g.index] = v.where(ok)
    return out


def seasonal_at(times, prof):
    key = pd.MultiIndex.from_arrays([times.dt.hour.to_numpy(), (times.dt.dayofweek >= 5).to_numpy()])
    return prof.reindex(key).fillna(1.0).to_numpy()


d = df.copy()
fwd = sum(grid_shift(d, "r2", -k) for k in range(1, 7))
d["y"] = 0.5 * np.log(fwd.where(fwd > 0) / 6 * PERIODS_PER_YEAR)
d["x_rv1h"] = d["log_rv1h"]
d["x_rv30"] = np.log(pd.to_numeric(d["realized_vol_30m_annualized"], errors="coerce").where(lambda s: s > 0))
d["x_abs_r"] = d["log_abs_r"]
d["x_weekend"] = d["is_weekend"].astype(float)

rows, clock_coefs = [], []
for asset, g in d.groupby("underlying"):
    tr_prof_src = g[g["is_train"] & g["r2"].notna()]
    prof = tr_prof_src.groupby(["hour", "is_weekend"])["r2"].mean() / tr_prof_src["r2"].mean()
    prof.index = pd.MultiIndex.from_arrays([prof.index.get_level_values(0).astype(int), prof.index.get_level_values(1).astype(bool)])
    st = g["state_time"]
    g = g.assign(
        x_s_fwd=np.log(np.mean([seasonal_at(st + pd.Timedelta(minutes=5 * i), prof) for i in range(1, 7)], axis=0)),
        x_s_1h=np.log(np.mean([seasonal_at(st - pd.Timedelta(minutes=5 * i), prof) for i in range(0, 12)], axis=0)),
    )
    M = {"M0": ["x_rv1h", "x_rv30", "x_abs_r"],
         "M0+clock": ["x_rv1h", "x_rv30", "x_abs_r", "x_s_fwd", "x_s_1h"],
         "M0+weekend": ["x_rv1h", "x_rv30", "x_abs_r", "x_weekend"]}
    g = g.dropna(subset=["y"] + M["M0+clock"])
    tr, te = g[g["is_train"]], g[~g["is_train"]]
    if len(tr) < 200 or len(te) < 200:
        continue
    row = {"underlying": asset, "n_train": len(tr), "n_test": len(te)}
    for name, xs in M.items():
        pred, b = ols_fit_predict(tr["y"].to_numpy(), tr[xs].to_numpy(), te[xs].to_numpy())
        row[f"oos_r2_{name}"] = oos_r2(te["y"].to_numpy(), pred, tr["y"].mean())
        if name == "M0+clock":
            clock_coefs.append({"underlying": asset, **dict(zip(["const"] + xs, b))})
    rows.append(row)
clock_forecast = pd.DataFrame(rows).set_index("underlying")
if len(clock_forecast):
    clock_forecast["gain_clock"] = clock_forecast["oos_r2_M0+clock"] - clock_forecast["oos_r2_M0"]
    clock_forecast["gain_weekend"] = clock_forecast["oos_r2_M0+weekend"] - clock_forecast["oos_r2_M0"]
display(clock_forecast.round(4))
print("Training coefficients of M0+clock:")
display(pd.DataFrame(clock_coefs).set_index("underlying").round(3))

# Part D — Options and the Clock

## 12. Do 0DTE options price the intraday pattern?

The settlement price is the 30-minute index TWAP 07:30–08:00. The variance of an average of a Brownian path over a window is one third of the variance accumulated over that window, so the expected variance of the log settlement price seen from $t$ is approximately

$$
\mathbb{E}_t[V_{t\to T}] = \sum_{u \in (t,\,T-30]} m_u \;+\; \tfrac13 \sum_{u \in (T-30,\,T]} m_u ,
$$

where $m_u$ is the expected squared return of bar $u$. Two "fair" volatilities follow, quoted like Deribit IV over the full $\tau_t$ (in bars, $n_t$):

$$
\sigma^{\text{seas}}_t = \sqrt{\frac{\mathrm{PERIODS\_PER\_YEAR}}{n_t}\,\mathbb{E}_t[V]} \quad \text{with } m_u = m_{h(u),w(u)}, \qquad
\sigma^{\text{flat}}_t \text{ with } m_u = \bar m .
$$

If the market prices the intraday pattern, $\log(\sigma_{IV}/\sigma^{\text{seas}})$ is **flatter across hours** than $\log(\sigma_{IV}/\sigma^{\text{flat}})$, and the hourly shapes of $\sigma_{IV}$ and $\sigma^{\text{seas}}$ agree. The level difference is the premium (next section); the shape is what matters here. The seasonal profile is in-sample. States in the settlement window and with less than one hour to expiry are excluded; IV is the quote-backed ATM IV.

In [ ]:
def settle_weight(times):
    mod = times.hour * 60 + times.minute
    return np.where((mod >= 455) & (mod <= 480), SETTLE_WEIGHT, 1.0)


def remaining_sum(g, values_on_grid, grid):
    """Sum of grid values over bars t+1..t+n (n = remaining bars) for every state of g."""
    C = np.concatenate([[0.0], np.cumsum(values_on_grid)])
    pos = ((g["state_time"] - grid[0]) / pd.Timedelta(minutes=5)).astype(int).to_numpy()
    n = (g["minutes_to_expiry"].to_numpy() // 5).astype(int)
    end = np.minimum(pos + n + 1, len(C) - 1)
    return C[end] - C[np.minimum(pos + 1, len(C) - 1)], n


opt_rows = []
for asset, g in df.groupby("underlying"):
    src = g[g["r2"].notna()]
    m_hw = src.groupby(["hour", "is_weekend"])["r2"].mean()
    m_hw.index = pd.MultiIndex.from_arrays([m_hw.index.get_level_values(0).astype(int), m_hw.index.get_level_values(1).astype(bool)])
    m_bar = src["r2"].mean()
    grid = pd.date_range(g["state_time"].min(), g["state_time"].max() + pd.Timedelta(days=1), freq="5min")
    w = settle_weight(grid)
    key = pd.MultiIndex.from_arrays([grid.hour, grid.dayofweek >= 5])
    m_grid = m_hw.reindex(key).fillna(m_bar).to_numpy()
    E_seas, n = remaining_sum(g, m_grid * w, grid)
    E_flat, _ = remaining_sum(g, m_bar * w, grid)
    gg = g.assign(n_bars=n,
                  fair_seas=100 * np.sqrt(E_seas / np.maximum(n, 1) * PERIODS_PER_YEAR),
                  fair_flat=100 * np.sqrt(E_flat / np.maximum(n, 1) * PERIODS_PER_YEAR),
                  iv=pd.to_numeric(g["option_iv_atm_forward"], errors="coerce"))
    gg = gg[gg["option_atm_quote_ok"] & ~gg["in_settlement_window"] & (gg["minutes_to_expiry"] >= 60) & (gg["iv"] > 0)]
    opt_rows.append(gg)
opt = pd.concat(opt_rows) if opt_rows else pd.DataFrame()
opt["log_iv_seas"] = np.log(opt["iv"] / opt["fair_seas"])
opt["log_iv_flat"] = np.log(opt["iv"] / opt["fair_flat"])

by_hour = opt.groupby(["underlying", "hour"]).agg(states=("iv", "size"), iv=("iv", "median"), fair_seas=("fair_seas", "median"),
                                                  log_iv_seas=("log_iv_seas", "median"), log_iv_flat=("log_iv_flat", "median"))
shape = by_hour.groupby("underlying").agg(
    hours=("states", "size"),
    std_across_hours_vs_seasonal=("log_iv_seas", "std"),
    std_across_hours_vs_flat=("log_iv_flat", "std"),
    median_log_iv_over_seasonal=("log_iv_seas", "median"),
)
shape["spearman_iv_vs_seasonal_shape"] = [by_hour.loc[a, "iv"].corr(by_hour.loc[a, "fair_seas"], method="spearman") for a in shape.index]
display(shape.round(3))
print("Hourly medians (IV and fair volatilities in %):")
display(by_hour.round(3).unstack(0)[["iv", "fair_seas", "log_iv_seas"]])

if CORE:
    fig, axes = plt.subplots(1, len(CORE), figsize=(6 * len(CORE), 3.4), squeeze=False)
    for ax, a in zip(axes[0], CORE):
        if a not in by_hour.index.get_level_values(0):
            continue
        b = by_hour.loc[a]
        ax.plot(b.index, b["iv"], marker="o", ms=3, label="ATM IV (median)")
        ax.plot(b.index, b["fair_seas"], marker="o", ms=3, label="seasonal fair vol")
        ax.plot(b.index, opt[opt["underlying"] == a]["fair_flat"].median() * np.ones(len(b)), ls="--", label="flat fair vol")
        ax.set_title(f"{a}: implied vs seasonal remaining volatility"); ax.set_xlabel("UTC hour"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 13. Premium by entry time, settlement-adjusted

Realized remaining variance uses the same one-third weight on the settlement window:

$$
V^{R}_{t\to T} = \sum_{u \in (t,\,T-30]} r_u^2 + \tfrac13 \sum_{u \in (T-30,\,T]} r_u^2, \qquad
\sigma^{R}_t = \sqrt{\frac{\mathrm{PERIODS\_PER\_YEAR}}{n_t} V^R_{t\to T}}
$$

(missing bars are scaled up when at least 90% of the weighted bars exist). One entry per session per full hour, grouped in four-hour blocks. The table also scores the two forecasts of $\log \sigma^R$, the market IV and the in-sample seasonal fair volatility, by mean squared error; the seasonal one has seen the data, so it is an optimistic benchmark. With about 15 sessions per block, these are descriptive.

In [ ]:
rv_rows = []
for asset, g in df.groupby("underlying"):
    grid = pd.date_range(g["state_time"].min(), g["state_time"].max() + pd.Timedelta(days=1), freq="5min")
    w = settle_weight(grid)
    r2g = g.set_index("state_time")["r2"].reindex(grid).to_numpy()
    present = np.isfinite(r2g)
    E_obs, n = remaining_sum(g, np.where(present, r2g, 0.0) * w, grid)
    W_obs, _ = remaining_sum(g, present * w, grid)
    W_all, _ = remaining_sum(g, w, grid)
    ok = W_obs >= 0.9 * W_all
    E = np.where(ok & (W_obs > 0), E_obs * W_all / np.where(W_obs > 0, W_obs, 1), np.nan)
    rv_rows.append(pd.Series(100 * np.sqrt(E / np.maximum(n, 1) * PERIODS_PER_YEAR), index=g.index))
df["rv_to_settle"] = pd.concat(rv_rows)

ent = opt[(opt["minute_of_day"] % 60 == 0)].join(df["rv_to_settle"])
ent = ent[ent["rv_to_settle"] > 0].copy()
ent["log_iv_rv"] = np.log(ent["iv"] / ent["rv_to_settle"])
ent["log_seas_rv"] = np.log(ent["fair_seas"] / ent["rv_to_settle"])
BLOCKS = {**{h: "08–12" for h in range(8, 12)}, **{h: "12–16" for h in range(12, 16)}, **{h: "16–20" for h in range(16, 20)},
          **{h: "20–24" for h in range(20, 24)}, **{h: "00–04" for h in range(0, 4)}, **{h: "04–07" for h in range(4, 8)}}
ent["block"] = pd.Categorical(ent["hour"].astype(int).map(BLOCKS), categories=["08–12", "12–16", "16–20", "20–24", "00–04", "04–07"], ordered=True)
premium_by_block = ent.groupby(["underlying", "block"], observed=True).agg(
    entries=("log_iv_rv", "size"), sessions=("session_key", "nunique"),
    median_log_iv_over_rv=("log_iv_rv", "median"), share_iv_above_rv=("log_iv_rv", lambda s: (s > 0).mean()),
    mse_log_iv=("log_iv_rv", lambda s: (s ** 2).mean()), mse_log_seasonal=("log_seas_rv", lambda s: (s ** 2).mean()),
)
display(premium_by_block.round(3))
print("Weekday vs weekend entries:")
display(ent.groupby(["underlying", "is_weekend"]).agg(entries=("log_iv_rv", "size"),
                                                     median_log_iv_over_rv=("log_iv_rv", "median")).round(3).unstack())

## 14. Quote support, liquidity and smile shape toward expiry

By time-to-expiry bucket and weekend: share of states with an accepted quote fit, with a quote-backed ATM node, the median share of OTM contracts within two expected moves that still have a bid, the fit error, and the fitted skew and curvature (relative to ATM IV). Because $\tau_t$ is the clock, these are the same profiles as by hour.

In [ ]:
agg = {"states": ("state_id", "size")}
if "fit_ok" in df:
    agg["fit_ok_share"] = ("fit_ok", "mean")
if "option_atm_quote_ok" in df:
    agg["atm_quote_ok_share"] = ("option_atm_quote_ok", "mean")
for c, name in [("otm_bid_share_z2", "median_otm_bid_share_z2"), ("fit_wrmse", "median_fit_wrmse")]:
    if c in df:
        agg[name] = (c, "median")
option_clock = df.groupby(["underlying", "tte_bucket", "is_weekend"], observed=True).agg(**agg)
fitted = df[df["fit_ok"]] if "fit_ok" in df else df.iloc[0:0]
shape_tte = fitted.groupby(["underlying", "tte_bucket"], observed=True).agg(
    median_skew_rel=("fit_skew_rel", "median"), median_curvature_rel=("fit_curvature_rel", "median"))
display(option_clock.round(3))
display(shape_tte.round(4).unstack(0))
eta_rows = []
for asset, g in fitted[~fitted["in_settlement_window"]].groupby("underlying"):
    eta_rows.append({"underlying": asset, "eta2_tte_skew": eta2(g["fit_skew_rel"], g["tte_bucket"]),
                     "eta2_tte_curvature": eta2(g["fit_curvature_rel"], g["tte_bucket"]),
                     "eta2_session_skew": eta2(g["fit_skew_rel"], g["session_key"]),
                     "eta2_session_curvature": eta2(g["fit_curvature_rel"], g["session_key"])})
display(pd.DataFrame(eta_rows).set_index("underlying").round(3))

# Part E — Calendar

## 15. Session-level volatility by weekday and expiry type

One row per session and asset: annualized session realized volatility from index returns and the first quote-backed ATM IV in the opening hour, both relative to the asset's median session ($\log$ ratio). Averaging the log ratios across assets gives one number per session (assets are strongly correlated, so they are not independent observations). With one quarterly and two or three weekly expiries in the sample, expiry-type differences cannot be separated from day-to-day variation yet.

In [ ]:
sess = df[df["r"].notna()].groupby(["underlying", "session_key"]).agg(
    bars=("r", "size"), var_sum=("r2", "sum"), expiry_type=("expiry_type", "first"), weekend_share=("is_weekend", "mean"))
sess["session_rv"] = 100 * np.sqrt(sess["var_sum"] / sess["bars"] * PERIODS_PER_YEAR)
open_iv = (df[df["option_atm_quote_ok"] & (df["minutes_to_expiry"] >= 1380)]
           .sort_values("state_time").groupby(["underlying", "session_key"])["option_iv_atm_forward"].first())
sess["open_iv"] = open_iv
sess = sess[sess["bars"] >= 200]
for c in ["session_rv", "open_iv"]:
    sess[f"log_rel_{c}"] = np.log(sess[c] / sess.groupby("underlying")[c].transform("median"))
per_session = sess.groupby("session_key").agg(
    expiry_type=("expiry_type", "first"), weekend_share=("weekend_share", "first"),
    mean_log_rel_rv=("log_rel_session_rv", "mean"), mean_log_rel_open_iv=("log_rel_open_iv", "mean"), assets=("bars", "size"))
per_session["expiry_weekday"] = [pd.Timestamp(s).day_name() for s in per_session.index]
display(per_session.round(3))
per_session["calendar"] = np.where(per_session["weekend_share"] > 0.5, "mostly weekend", "weekday")
print("By expiry type:")
display(per_session.groupby("expiry_type")[["mean_log_rel_rv", "mean_log_rel_open_iv"]].agg(["mean", "count"]).round(3))
print("By calendar:")
display(per_session.groupby("calendar")[["mean_log_rel_rv", "mean_log_rel_open_iv"]].agg(["mean", "count"]).round(3))

## 16. Interpretation checklist

| Result | If yes | If no |
|---|---|---|
| §4–5: $\eta^2$ of hour clearly above its bias, profile stable across halves and shared across assets | The structural scale must carry a deterministic diurnal factor: $\sigma_{k,t} = \sigma_k \sqrt{s(\tau_t, c_t)}$ or deseasonalized inputs | Clock baseline only for liquidity features |
| §6: weekend variance ratio interval excludes 1 | Weekend term in $c_t$ for volatility | Weekend only for liquidity features |
| §7: large $\eta^2$ for volume/updates/spread, small for coherence noise | Activity needs a clock baseline (relevant for any later noise input); coherence noise does not | — |
| §8: $R^2_{\text{day}}$ large relative to $R^2_{\text{clock}}$ | Much of volatility variation is slow (day-level): the HSMM's volatility states may be day-length, not intraday | Intraday variation dominates |
| §9: ACF at 3–12 h and 288 bars falls after deseasonalization | Raw persistence is partly clock; durations must be clock-centred (v2.1 §18.2) and inputs deseasonalized before fitting | Persistence is volatility clustering |
| §10: a low-order Fourier model (F2–F4) is within ~0.01 of H24 out of sample | Use the smooth Fourier baseline in `src` (Step 2) | Hour dummies (or a spline) needed |
| §11: positive `gain_clock`, coefficient on $\log \bar s^{fwd}$ near 0.5 | The diurnal factor is a forecasting input, not just a description | — |
| §12: IV tracks seasonal fair vol (flat $\log(\sigma_{IV}/\sigma^{seas})$, high shape correlation) | Options price the clock; premium must be measured against $\sigma^{seas}$, not a flat RV | IV ignores the pattern: an hour-dependent mispricing candidate for the volatility book, to be confirmed with more sessions |
| §13: premium differs by entry block | Entry timing is part of the volatility-book design | — |
| §14: curvature driven by $\tau$, skew by session | Curvature needs a clock baseline; skew is a state variable | — |

In [ ]:
clock_report = {
    "variance_profile": variance_profile.reset_index(),
    "profile_ci": profile_ci,
    "profile_stability": profile_stability.reset_index(),
    "weekend_ratio": weekend_ratio.reset_index(),
    "liquidity_clock": liquidity_clock,
    "variance_decomposition": variance_decomposition.reset_index(),
    "acf_table": acf_table.reset_index(),
    "baseline_oos": baseline_oos,
    "clock_forecast": clock_forecast.reset_index(),
    "iv_shape": shape.reset_index(),
    "premium_by_block": premium_by_block.reset_index(),
    "option_clock": option_clock.reset_index(),
    "per_session": per_session.reset_index(),
}
for k, val in clock_report.items():
    print(f"  {k:<24} {len(val):>7,} rows")

## 17. Close the read-only connection

In [ ]:
con.close()
print("DuckDB connection closed.")